# Kepler exoplanet classification — Notebook 04

**Final report: evaluation, interpretability, and comparison with prior work**

Author: Atilla Ahmed

---

## Abstract

This notebook consolidates the results from the previous three notebooks into a final scientific report. We evaluate the best deep learning model (FT-Transformer) and the best classical baseline (XGBoost) on the held-out test set, analyze model interpretability through SHAP feature importance, assess calibration quality, and compare our results against published prior work on the Kepler KOI dataset. All results are reported under the leak-free feature configuration (Setup C) to ensure honest performance estimates.

## Table of contents

1. [Setup and model loading](#1-setup-and-model-loading)
2. [Final leaderboard](#2-final-leaderboard)
3. [Test set evaluation](#3-test-set-evaluation)
4. [Interpretability](#4-interpretability)
5. [Calibration analysis](#5-calibration-analysis)
6. [Comparison with prior work](#6-comparison-with-prior-work)
7. [Conclusions and limitations](#7-conclusions-and-limitations)

## 1. Setup and model loading

We load the processed data, the trained model checkpoints from Notebook 03, and the baseline results from Notebook 02. This notebook does not train any new models — it operates purely on saved artifacts.

### 1.1. Imports and configuration

In [1]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import joblib
import shap
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
    brier_score_loss,
)

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_SEED = 42
BATCH_SIZE = 256

PROCESSED_PATH = Path("../data/processed")
BASELINES_PATH = Path("../models/baselines")
DL_PATH = Path("../models/dl")
FIGURES_PATH = Path("../reports/figures")
FIGURES_PATH.mkdir(parents=True, exist_ok=True)

int_to_class = {0: "CONFIRMED", 1: "CANDIDATE", 2: "FALSE POSITIVE"}
class_order = ["CONFIRMED", "CANDIDATE", "FALSE POSITIVE"]
colors = ["green", "orange", "red"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"Device: {device}")
print("Setup complete")

Device: cuda
Setup complete


### 1.2. Load data, baselines, and trained models

In [2]:
train_df = pd.read_parquet(PROCESSED_PATH / "train.parquet")
val_df   = pd.read_parquet(PROCESSED_PATH / "val.parquet")
test_df  = pd.read_parquet(PROCESSED_PATH / "test.parquet")

with open(PROCESSED_PATH / "feature_sets.json", "r") as f:
    feature_sets = json.load(f)

feature_sets_effective = {
    "setup_leaky":     [c for c in feature_sets["setup_a_leaky"] if c in train_df.columns and c != "target"],
    "setup_leak_free": [c for c in feature_sets["setup_c_leak_free"] if c in train_df.columns and c != "target"],
}

features_c = feature_sets_effective["setup_leak_free"]

def split_features_target(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    return df.drop(columns=["target"]), df["target"]

X_train, y_train = split_features_target(train_df)
X_val,   y_val   = split_features_target(val_df)
X_test,  y_test  = split_features_target(test_df)

baseline_results = pd.read_csv(BASELINES_PATH / "results.csv")
xgb_pipeline = joblib.load(BASELINES_PATH / "xgboost_setup_leak_free.joblib")

print(f"Data loaded: train={X_train.shape}, val={X_val.shape}, test={X_test.shape}")
print(f"Leak-free features: {len(features_c)}")
print(f"Baseline results:\n{baseline_results.to_string(index=False)}")

Data loaded: train=(6694, 101), val=(1435, 101), test=(1435, 101)
Leak-free features: 51
Baseline results:
  model           setup  n_features  accuracy  macro_f1  f1_confirmed  f1_candidate  f1_false_pos
XGBoost setup_leak_free          51  0.837631  0.798671      0.888626      0.613191      0.894198
 LogReg setup_leak_free          51  0.758885  0.690807      0.799562      0.425703      0.847156
XGBoost     setup_leaky          55  0.940070  0.918727      0.906921      0.856164      0.993094
 LogReg     setup_leaky          55  0.868990  0.834226      0.803167      0.735552      0.963958
